In [1]:
import pandas as pd
import pandas as pd
from sklearn.model_selection import train_test_split
from xgboost import XGBRegressor
train_df = pd.read_csv("train.csv")
test_df = pd.read_csv("test.csv")
print(train_df.head())
print(test_df.head())


       Timestamp  Residents Apartment_Type  Temperature Humidity  Water_Price  \
0  01/01/2002 00          1         Studio        15.31    46.61         1.06   
1  01/01/2002 08          4            NaN        21.01    66.11         2.98   
2  01/01/2002 16          2        Cottage        12.86    60.86         1.44   
3  02/01/2002 00          2           1BHK        20.16    50.58         1.48   
4  02/01/2002 08          2        Cottage        16.23    52.25         1.14   

   Period_Consumption_Index  Income_Level  Guests      Amenities  \
0                      0.97           Low       0  Swimming Pool   
1                      0.91  Upper Middle       1  Swimming Pool   
2                      1.43        Middle       0            NaN   
3                      0.91        Middle      -1         Garden   
4                      1.11        Middle       0       Fountain   

   Appliance_Usage  Water_Consumption  
0              0.0              64.85  
1              1.0      

In [2]:
import numpy as np
train_df['Humidity'] = pd.to_numeric(train_df['Humidity'], errors='coerce')
test_df['Humidity'] = pd.to_numeric(test_df['Humidity'], errors='coerce')

# Replace invalid values (-99, -2) with NaN
invalid_values = {-99: np.nan, -2: np.nan}
train_df.replace(invalid_values, inplace=True)
test_df.replace(invalid_values, inplace=True)

fill_values = {
    'Apartment_Type': train_df['Apartment_Type'].mode()[0],
    'Temperature': train_df['Temperature'].median(),
    'Income_Level': train_df['Income_Level'].mode()[0],
    'Appliance_Usage': train_df['Appliance_Usage'].median(),
    'Residents': train_df['Residents'].median(),
    'Water_Price': train_df['Water_Price'].median(),
    'Guests': train_df['Guests'].median()
}
train_df.fillna(fill_values, inplace=True)
test_df.fillna(fill_values, inplace=True)


In [3]:
train_df['Timestamp'] = pd.to_datetime(train_df['Timestamp'], format="%d/%m/%Y %H")
test_df['Timestamp'] = pd.to_datetime(test_df['Timestamp'], format="%d/%m/%Y %H")


for df in [train_df, test_df]:
    df['Hour'] = df['Timestamp'].dt.hour
    df['Day'] = df['Timestamp'].dt.day
    df['Month'] = df['Timestamp'].dt.month
    df['Weekday'] = df['Timestamp'].dt.weekday
train_df.drop(columns=['Timestamp'], inplace=True)
test_df.drop(columns=['Timestamp'], inplace=True)


In [4]:
# Convert categorical columns to numerical using one-hot encoding
categorical_cols = ['Apartment_Type', 'Income_Level', 'Amenities']
train_df = pd.get_dummies(train_df, columns=categorical_cols, drop_first=True)
test_df = pd.get_dummies(test_df, columns=categorical_cols, drop_first=True)

# Ensure test set has the same columns as train (excluding target variable)
missing_cols = set(train_df.columns) - set(test_df.columns) - {'Water_Consumption'}
for col in missing_cols:
    test_df[col] = 0

test_df = test_df[train_df.drop(columns=['Water_Consumption']).columns]


<ipython-input-4-809aab22057c>:9: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  test_df[col] = 0
<ipython-input-4-809aab22057c>:9: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  test_df[col] = 0
<ipython-input-4-809aab22057c>:9: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  test_df[col] = 0
<ipython-i

In [5]:
train_df = pd.read_csv("train.csv")

# Ensure column names are valid for XGBoost before one-hot encoding
train_df.columns = train_df.columns.str.replace(r"[^a-zA-Z0-9_]", "_", regex=True)
categorical_cols = train_df.select_dtypes(include=["object"]).columns

# Convert categorical columns using one-hot encoding
train_df = pd.get_dummies(train_df, columns=categorical_cols, drop_first=True)
train_df.columns = train_df.columns.str.replace(r"[^a-zA-Z0-9_]", "_", regex=True)

cols_to_drop = [col for col in ["Water_Consumption", "Timestamp"] if col in train_df.columns]
X = train_df.drop(columns=cols_to_drop)
y = train_df["Water_Consumption"]

# Convert DataFrame to NumPy array before training
X = X.values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Fit the model
xgb_model = XGBRegressor(n_estimators=100, learning_rate=0.1, random_state=42)
xgb_model.fit(X_train, y_train)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=None, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.1, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=None,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=100,
             n_jobs=None, num_parallel_tree=None, ...)

In [6]:
print(train_df.columns)


Index(['Residents', 'Temperature', 'Water_Price', 'Period_Consumption_Index',
       'Guests', 'Appliance_Usage', 'Water_Consumption',
       'Timestamp_01_01_2002_08', 'Timestamp_01_01_2002_16',
       'Timestamp_01_01_2003_00',
       ...
       'Income_Level__zP9_', 'Income_Level___VYi', 'Income_Level___f_U',
       'Income_Level___Wo_', 'Income_Level___XT_', 'Income_Level__U_o7',
       'Income_Level__lWKR', 'Amenities_Garden', 'Amenities_Jacuzzi',
       'Amenities_Swimming_Pool'],
      dtype='object', length=18948)


In [7]:
import pandas as pd
from sklearn.model_selection import train_test_split
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Load datasets
train_df = pd.read_csv("train.csv")
test_df = pd.read_csv("test.csv")

# Ensure column names are valid for XGBoost before one-hot encoding
train_df.columns = train_df.columns.str.replace(r"[^a-zA-Z0-9_]", "_", regex=True)
test_df.columns = test_df.columns.str.replace(r"[^a-zA-Z0-9_]", "_", regex=True)

categorical_cols_train = train_df.select_dtypes(include=["object"]).columns
categorical_cols_test = test_df.select_dtypes(include=["object"]).columns

In [8]:
import pandas as pd
from sklearn.model_selection import train_test_split
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

train_df = pd.read_csv("train.csv")
test_df = pd.read_csv("test.csv")

train_df.columns = train_df.columns.str.replace(r"[^a-zA-Z0-9_]", "_", regex=True)
test_df.columns = test_df.columns.str.replace(r"[^a-zA-Z0-9_]", "_", regex=True)

categorical_cols_train = train_df.select_dtypes(include=["object"]).columns
categorical_cols_test = test_df.select_dtypes(include=["object"]).columns

train_df = pd.get_dummies(train_df, columns=categorical_cols_train, drop_first=True)
test_df = pd.get_dummies(test_df, columns=categorical_cols_test, drop_first=True)

missing_cols = set(train_df.columns) - set(test_df.columns) - {'Water_Consumption'}
for col in missing_cols:
    test_df[col] = 0

test_df = test_df[[c for c in train_df.columns if c != 'Water_Consumption']]

test_predictions = xgb_model.predict(test_df)

Streaming output truncated to the last 5000 lines.
<ipython-input-8-a518affef415>:20: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  test_df[col] = 0
<ipython-input-8-a518affef415>:20: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  test_df[col] = 0
<ipython-input-8-a518affef415>:20: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `

In [9]:
test_df = pd.read_csv("test.csv")
print(test_df["Timestamp"].head())
print(len(test_predictions), len(test_df))


0    11/10/2014 16
1    12/10/2014 00
2    12/10/2014 08
3    12/10/2014 16
4    13/10/2014 00
Name: Timestamp, dtype: object
6000 6000


In [10]:

submission = pd.DataFrame({
    "Timestamp": test_df["Timestamp"],
    "Water_Consumption": test_predictions
})


submission.to_csv("submission.csv", index=False)

# Verify file structure
print(submission.head())



       Timestamp  Water_Consumption
0  11/10/2014 16         318.345856
1  12/10/2014 00         207.160233
2  12/10/2014 08          89.779800
3  12/10/2014 16         128.397385
4  13/10/2014 00         127.573174
